# 集合覆盖 scp41（最小成本）

**问题**：实例来自 OR-Library 的 scp41：m=200 个行元素，n=1000 个列集合。列 j 的成本为 c_j，覆盖的行集合为 S_j（a_ij=1 表示列 j 覆盖行 i）。目标是选择一组列，使每一行至少被一个选中列覆盖，同时总成本最小。

**数学模型**

$$\min \sum_{j=1}^{n} c_j x_j$$

$$\text{s.t.}\quad \sum_{j: i \in S_j} x_j \ge 1,\quad i=1,\dots,m$$

$$x_j\in\{0,1\},\quad j=1,\dots,n$$

数据文件：\`/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt\`。文献最优值 429（本套件用直接 MIP 自证）。

## 方法：Benders 分解（LP 松弛的切割平面变体）

**适配说明**：经典 SCP 没有天然的连续 recourse 子问题，因此这里把 Benders 作用于 **LP 松弛**：主问题用二元 y_j 表示“允许使用哪些列”，子问题在 y 固定的子集上解连续覆盖 LP。为避免主问题目标中列成本被重复计算，主问题目标取 \`min θ\`（列成本已包含在子问题中）。

**主问题 MP**

$$\min \theta$$

$$\text{s.t.}\quad \theta + \sum_j \lambda_j^k y_j \ge \sum_i \pi_i^k,\quad k=1,\dots,K$$

$$y_j\in\{0,1\},\quad \theta\ge 0$$

**子问题 SP(y)**（带人工变量 s，始终可行）

$$\min \sum_j c_j x_j + M\sum_i s_i$$

$$\text{s.t.}\quad \sum_{j: i\in S_j} x_j + s_i \ge 1,\quad i=1,\dots,m$$

$$0\le x_j \le y_j,\quad s_i\ge 0$$

对偶最优解 $\pi_i^k$（行约束）与 $\lambda_j^k$（上界约束的标准非负乘子）给出最优性割。

**原理要点**

1. SP 是连续 LP，用 GLOP 求对偶；MathOpt 对 \`x<=y\` 约束返回的非正对偶需要取反才是标准 $\lambda\ge 0$。
2. 每次解 MP（HIGHS MIP）得 y 与 θ；对当前 y 解 SP 生成割。
3. MP 目标 min θ 被割逐次抬高，收敛到 LP 松弛下界。
4. 停机：y 不再变化、迭代上限 20、总墙钟 110s。
5. 收敛后仍用 HIGHS 在完整池上解整数 MIP 修复。

In [1]:
import platform, time, datetime, math, ortools
from ortools.math_opt.python import mathopt

print("python", platform.python_version(), "| ortools", ortools.__version__)

DATA = "/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt"
toks = open(DATA).read().split()
m, n = map(int, toks[:2])
costs = list(map(int, toks[2:2+n]))
idx = 2 + n
rows = []
for _ in range(m):
    k = int(toks[idx]); idx += 1
    rows.append([int(t)-1 for t in toks[idx:idx+k]]); idx += k
assert idx == len(toks)
colrows = [[] for _ in range(n)]
for i, row in enumerate(rows):
    for j in row:
        colrows[j].append(i)
print("m,n =", m, n, "| rows parsed =", len(rows), "| tokens consumed =", idx)


python 3.10.20 | ortools 9.15.6755
m,n = 200 1000 | rows parsed = 200 | tokens consumed = 5211


In [2]:
M = float(sum(costs) + 1)
lp_params = mathopt.SolveParameters(enable_output=False)
mip_params = mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=20), enable_output=False)

def solve_sp(y):
    sp = mathopt.Model(name="sp")
    s = [sp.add_variable(lb=0.0, ub=float('inf'), is_integer=False, name=f"s{i}") for i in range(m)]
    x = [sp.add_variable(lb=0.0, ub=float('inf'), is_integer=False, name=f"x{j}") for j in range(n)]
    row_cons = []
    for i, row in enumerate(rows):
        row_cons.append(sp.add_linear_constraint(sum(x[j] for j in row) + s[i] >= 1.0, name=f"cov{i}"))
    xub = [sp.add_linear_constraint(x[j] <= float(y[j]), name=f"xub{j}") for j in range(n)]
    sp.minimize_linear_objective(sum(costs[j]*x[j] for j in range(n)) + M*sum(s[i] for i in range(m)))
    res = mathopt.solve(sp, mathopt.SolverType.GLOP, params=lp_params)
    pi = res.dual_values(row_cons)
    mu = res.dual_values(xub)
    if not isinstance(pi, list): pi = [pi[c] for c in row_cons]
    if not isinstance(mu, list): mu = [mu[c] for c in xub]
    alpha = sum(pi)
    lam = [-v for v in mu]  # MathOpt <= dual is nonpositive; convert to standard lambda >= 0
    return alpha, lam, res.objective_value()

def solve_master(cuts):
    mp = mathopt.Model(name="master")
    y = [mp.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"y{j}") for j in range(n)]
    theta = mp.add_variable(lb=0.0, ub=float('inf'), is_integer=False, name="theta")
    for kk, (alpha, lam) in enumerate(cuts):
        mp.add_linear_constraint(theta + sum(lam[j]*y[j] for j in range(n)) >= alpha, name=f"bcut{kk}")
    mp.minimize_linear_objective(theta)
    res = mathopt.solve(mp, mathopt.SolverType.HIGHS, params=mip_params)
    yv = res.variable_values(y)
    ystar = [1 if yv[j] > 0.5 else 0 for j in range(n)]
    th = res.variable_values([theta])[0]
    return ystar, th, res.objective_value(), res.termination.reason

t0 = time.perf_counter()
current = [1]*n
cuts = []
max_iter = 20
for it in range(max_iter):
    alpha, lam, sp_obj = solve_sp(current)
    cuts.append((alpha, lam))
    ystar, theta, mp_obj, mp_term = solve_master(cuts)
    print(f"iter {it+1}: SP_obj={sp_obj:.4f}, MP_obj={mp_obj:.4f}, theta={theta:.4f}, selected={sum(ystar)}, cuts={len(cuts)}, MP_term={mp_term}")
    if ystar == current:
        print("master solution unchanged -> convergence")
        break
    current = ystar
    if time.perf_counter()-t0 > 110:
        print("time limit reached")
        break
wall = time.perf_counter()-t0
print("Benders wall:", round(wall, 3), "| iterations:", it+1, "| cuts:", len(cuts), "| Benders lower bound:", mp_obj)

# integer repair on full pool
t1 = time.perf_counter()
mip = mathopt.Model(name="scp41_benders_repair")
x = [mip.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"x{j}") for j in range(n)]
mip.minimize_linear_objective(sum(costs[j]*x[j] for j in range(n)))
for i, row in enumerate(rows):
    mip.add_linear_constraint(sum(x[j] for j in row) >= 1.0, name=f"cov{i}")
mres = mathopt.solve(mip, mathopt.SolverType.HIGHS, params=mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=120), enable_output=False))
mv = mres.variable_values(x)
sel = [j for j in range(n) if mv[j] > 0.5]
print("integer repair:", mres.termination.reason, "| obj:", mres.objective_value(), "| best_bound:", mres.best_objective_bound(), "| wall:", round(time.perf_counter()-t1, 3))
print("selected_columns:", sorted(sel))


iter 1: SP_obj=429.0000, MP_obj=429.0000, theta=429.0000, selected=19, cuts=1, MP_term=TerminationReason.OPTIMAL
iter 2: SP_obj=6656877.0000, MP_obj=429.0000, theta=429.0000, selected=997, cuts=2, MP_term=TerminationReason.OPTIMAL


iter 3: SP_obj=429.0000, MP_obj=429.0000, theta=429.0000, selected=997, cuts=3, MP_term=TerminationReason.OPTIMAL
master solution unchanged -> convergence
Benders wall: 0.828 | iterations: 3 | cuts: 3 | Benders lower bound: 429.0
integer repair: TerminationReason.OPTIMAL | obj: 429.0 | best_bound: 429.0 | wall: 0.153
selected_columns: [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 24, 25, 27, 28, 42, 43, 45, 46, 47, 48, 49, 51, 53, 57, 58, 61, 62, 65, 68, 69, 70, 74, 76, 77, 80, 84, 85, 88, 90, 93, 102, 106, 115, 119, 120, 121, 123, 128, 137, 142, 143, 145, 152, 193, 274, 432]


## 运行结果与结论

上方输出显示：Benders 主问题在第 3 轮后 y 不变，得到 LP 松弛下界 **429.0**；完整池整数 MIP 修复得到并证明 **429.0**。

**基准最优值来源**：直接 MIP（01_direct）证明最优值 429.0；Benders 下界与整数修复上界均为 429.0。

## 结论

对纯覆盖问题，Benders 没有天然连续 recourse，本实现是“二元选列主问题 + 连续覆盖 LP 子问题”的可运行变体，3 轮即得到 LP 下界 429，但整体不如直接 MIP/LBBD 自然；其定位是展示对偶割机制。